# 기존 코드

In [18]:
import os
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from imblearn.over_sampling import ADASYN
from sklearn.model_selection import train_test_split
from sklearn.ensemble import BaggingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import f1_score
from category_encoders import BinaryEncoder
import lightgbm as lgb
from sklearn.ensemble import VotingClassifier
from catboost import CatBoostClassifier


In [19]:
ROOT_DIR = "data"
RANDOM_STATE = 110

# 데이터 로드
train_data = pd.read_csv(os.path.join(ROOT_DIR, "train.csv"))
test_data = pd.read_csv(os.path.join(ROOT_DIR, "test.csv"))

# test_data에서 id 열 저장
test_ids = test_data['Set ID']

# target 컬럼 분리
target = train_data['target']

# train data와 test data를 concat
concated = pd.concat([train_data.drop(columns=['target']), test_data.drop(columns=['Set ID'])], ignore_index=True)
df = concated

selected_columns = [
'Production Qty Collect Result_Fill1',
'Chamber Temp. Judge Value_AutoClave',
'Equipment_Fill1',
'HEAD NORMAL COORDINATE Z AXIS(Stage2) Collect Result_Fill1',
'Head Clean Position Y Collect Result_Dam',
'Head Purge Position X Collect Result_Fill1',
'HEAD Standby Position X Collect Result_Fill2',
'DISCHARGED TIME OF RESIN(Stage3) Collect Result_Fill1',
'Stage2 Circle1 Distance Speed Collect Result_Dam',
'HEAD Standby Position Y Collect Result_Fill1',
'Machine Tact time Collect Result_Fill2',
'PalletID Collect Result_Fill2',
'WorkMode Collect Result_Dam',
'Head Clean Position X Collect Result_Dam',
'Head Purge Position Z Collect Result_Dam',
'Equipment_Dam',
'Head Zero Position X Collect Result_Dam',
'Receip No Collect Result_Fill2',
'Stage3 Circle2 Distance Speed Collect Result_Dam',
'2nd Pressure Unit Time_AutoClave',
'Dispense Volume(Stage2) Collect Result_Dam',
'Stage3 Circle1 Distance Speed Collect Result_Dam',
'HEAD NORMAL COORDINATE X AXIS(Stage2) Collect Result_Fill1',
'CURE START POSITION X Collect Result_Fill2',
'THICKNESS 3 Collect Result_Dam',
'DISCHARGED TIME OF RESIN(Stage3) Collect Result_Dam',
'3rd Pressure Collect Result_AutoClave',
'Stage3 Line2 Distance Speed Collect Result_Dam',
'Stage2 Line2 Distance Speed Collect Result_Dam',
'CURE END POSITION Z Collect Result_Dam',
'HEAD NORMAL COORDINATE X AXIS(Stage3) Collect Result_Fill1',
'Head Purge Position Z Collect Result_Fill2',
'Dispense Volume(Stage1) Collect Result_Dam',
'Receip No Collect Result_Fill1',
'HEAD NORMAL COORDINATE X AXIS(Stage2) Collect Result_Dam',
'Stage3 Line4 Distance Speed Collect Result_Dam',
'PalletID Collect Result_Dam',
'HEAD NORMAL COORDINATE Y AXIS(Stage2) Collect Result_Fill1',
'Dispense Volume(Stage3) Collect Result_Dam',
'HEAD NORMAL COORDINATE X AXIS(Stage3) Collect Result_Dam',
'CURE START POSITION X Collect Result_Dam',
'Head Clean Position Z Collect Result_Fill2',
'HEAD NORMAL COORDINATE Y AXIS(Stage1) Collect Result_Fill1',
'DISCHARGED SPEED OF RESIN Collect Result_Fill1',
'HEAD NORMAL COORDINATE Z AXIS(Stage2) Collect Result_Dam',
'HEAD NORMAL COORDINATE Y AXIS(Stage3) Collect Result_Fill2',
'Head Purge Position Z Collect Result_Fill1',
'Head Clean Position Y Collect Result_Fill2',
'1st Pressure 1st Pressure Unit Time_AutoClave',
'Stage2 Line3 Distance Speed Collect Result_Dam',
'2nd Pressure Collect Result_AutoClave',
'CURE STANDBY POSITION Z Collect Result_Fill2',
'Head Purge Position X Collect Result_Dam',
'HEAD Standby Position X Collect Result_Dam',
'HEAD NORMAL COORDINATE Z AXIS(Stage1) Collect Result_Dam',
'1st Pressure Collect Result_AutoClave',
'HEAD Standby Position Y Collect Result_Fill2',
'Stage1 Line2 Distance Speed Collect Result_Dam',
'HEAD Standby Position Z Collect Result_Fill2',
'Head Zero Position Z Collect Result_Dam',
'Dispense Volume(Stage1) Collect Result_Fill1',
'HEAD Standby Position Y Collect Result_Dam',
'HEAD NORMAL COORDINATE Y AXIS(Stage3) Collect Result_Dam',
'Equipment_Fill2',
'HEAD NORMAL COORDINATE Y AXIS(Stage2) Collect Result_Fill2',
'Workorder_Dam',
'Production Qty Collect Result_Dam',
'HEAD NORMAL COORDINATE Y AXIS(Stage2) Collect Result_Dam',
'HEAD Standby Position Z Collect Result_Dam',
'DISCHARGED TIME OF RESIN(Stage1) Collect Result_Dam',
'DISCHARGED TIME OF RESIN(Stage2) Collect Result_Fill1',
'THICKNESS 1 Collect Result_Dam',
'WorkMode Collect Result_Fill2',
'Head Clean Position X Collect Result_Fill2',
'Machine Tact time Collect Result_Fill1',
'CURE END POSITION X Collect Result_Dam',
'HEAD NORMAL COORDINATE Y AXIS(Stage3) Collect Result_Fill1',
'Stage3 Line1 Distance Speed Collect Result_Dam',
'HEAD NORMAL COORDINATE X AXIS(Stage2) Collect Result_Fill2',
'DISCHARGED TIME OF RESIN(Stage1) Collect Result_Fill1',
'Model.Suffix_Dam',
'CURE END POSITION Z Collect Result_Fill2',
'CURE END POSITION X Collect Result_Fill2',
'HEAD NORMAL COORDINATE Z AXIS(Stage1) Collect Result_Fill1',
'HEAD Standby Position X Collect Result_Fill1',
'Chamber Temp. Unit Time_AutoClave',
'PalletID Collect Result_Fill1',
'CURE END POSITION Θ Collect Result_Dam',
'Machine Tact time Collect Result_Dam',
'Stage2 Line4 Distance Speed Collect Result_Dam',
'DISCHARGED SPEED OF RESIN Collect Result_Dam',
'HEAD NORMAL COORDINATE Y AXIS(Stage1) Collect Result_Dam',
'HEAD NORMAL COORDINATE Z AXIS(Stage3) Collect Result_Fill1',
'Head Clean Position X Collect Result_Fill1',
'Head Zero Position Y Collect Result_Dam',
'Receip No Collect Result_Dam',
'Stage1 Circle2 Distance Speed Collect Result_Dam',
'HEAD NORMAL COORDINATE Z AXIS(Stage2) Collect Result_Fill2',
'3rd Pressure Unit Time_AutoClave',
'Dispense Volume(Stage2) Collect Result_Fill1',
'Head Clean Position Y Collect Result_Fill1',
'Head Clean Position Z Collect Result_Fill1',
'CURE SPEED Collect Result_Fill2',
'HEAD NORMAL COORDINATE Y AXIS(Stage1) Collect Result_Fill2',
'Production Qty Collect Result_Fill2',
'Stage2 Circle2 Distance Speed Collect Result_Dam',
'Stage1 Circle1 Distance Speed Collect Result_Dam',
'Stage1 Line1 Distance Speed Collect Result_Dam',
'CURE SPEED Collect Result_Dam',
'HEAD NORMAL COORDINATE X AXIS(Stage3) Collect Result_Fill2',
'Stage1 Line4 Distance Speed Collect Result_Dam',
'Head Purge Position X Collect Result_Fill2',
'DISCHARGED TIME OF RESIN(Stage2) Collect Result_Dam',
'HEAD Standby Position Z Collect Result_Fill1',
'THICKNESS 2 Collect Result_Dam',
'WorkMode Collect Result_Fill1',
'HEAD NORMAL COORDINATE Z AXIS(Stage1) Collect Result_Fill2',
'Dispense Volume(Stage3) Collect Result_Fill1',
'Chamber Temp. Collect Result_AutoClave',
'Head Clean Position Z Collect Result_Dam',
]

df_selected = df[selected_columns]

# 원래 train 데이터와 test 데이터 분리
X_train = df_selected.iloc[:len(train_data), :]
X_test = df_selected.iloc[len(train_data):, :]

# Binary Encoding 수행
binary_encoder = BinaryEncoder(cols=X_train.select_dtypes(include=['object']).columns)
X_train_encoded = binary_encoder.fit_transform(X_train)
X_test_encoded = binary_encoder.transform(X_test)

# Target 변수도 인코딩
target_encoder = LabelEncoder()
y_train = target_encoder.fit_transform(target)

# ADASYN 오버샘플링 적용
adasyn = ADASYN(random_state=RANDOM_STATE)
X_resampled, y_resampled = adasyn.fit_resample(X_train_encoded, y_train)

# 학습 데이터와 검증 데이터로 분할
df_resampled = pd.concat([pd.DataFrame(X_resampled, columns=X_train_encoded.columns), pd.DataFrame(y_resampled, columns=['target'])], axis=1)
# df_train, df_val = train_test_split(
#     df_resampled,
#     test_size=0.28,
#     stratify=df_resampled["target"],
#     random_state=RANDOM_STATE,
# )

df_train = df_resampled

# BaggingClassifier 학습 및 예측
X_train_final = df_train.drop(columns=["target"])
y_train_final = df_train["target"]

# X_val = df_val.drop(columns=["target"])
# y_val = df_val["target"]

# CART bagging

In [20]:
cart_bagging_model_reduced = BaggingClassifier(estimator=DecisionTreeClassifier(random_state=42),
                                               n_estimators=55,random_state=42)
cart_bagging_model_reduced.fit(X_train_final, y_train_final)

# # 검증 데이터에서 예측 수행
# y_pred_cart_bagging_reduced = cart_bagging_model_reduced.predict(X_val)

BaggingClassifier(estimator=DecisionTreeClassifier(random_state=42),
                  n_estimators=55, random_state=42)

In [21]:
# # F1 Score 계산
# f1_cart_bagging_reduced = f1_score(y_val, y_pred_cart_bagging_reduced, average='binary', pos_label=0)
# print(f'Validation F1 Score: {f1_cart_bagging_reduced}')

# LGBM

In [22]:
# LightGBM 모델 설정
lgb_model = lgb.LGBMClassifier(random_state=RANDOM_STATE)

# 최적 파라미터 설정(Optuna 사용)
best_lgb_model = lgb.LGBMClassifier(
    num_leaves=366,
    max_depth=10,
    learning_rate=0.021711635900896083,
    n_estimators=800,
    class_weight='balanced',
    min_child_samples=5,
    subsample=0.7833554249870598,
    colsample_bytree=0.7612130831440033,
    reg_alpha=0.6540453950629128,
    reg_lambda=3.226039925574031,
    random_state=RANDOM_STATE  # 동일한 결과를 재현하기 위한 랜덤 시드 설정
)


best_lgb_model.fit(X_train_final, y_train_final)

# # 검증 데이터에서 예측 수행
# y_pred_lgb = best_lgb_model.predict(X_val)


[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 38156, number of negative: 37756
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.043631 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 15497
[LightGBM] [Info] Number of data points in the train set: 75912, number of used features: 135
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with posit

[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f

[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f

[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f

[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f

[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f

[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f

LGBMClassifier(class_weight='balanced', colsample_bytree=0.7612130831440033,
               learning_rate=0.021711635900896083, max_depth=10,
               min_child_samples=5, n_estimators=800, num_leaves=366,
               random_state=110, reg_alpha=0.6540453950629128,
               reg_lambda=3.226039925574031, subsample=0.7833554249870598)

In [23]:
# # F1 Score 계산
# f1 = f1_score(y_val, y_pred_lgb, average='binary', pos_label=0)
# print(f'Validation F1 Score: {f1}')

# CAT Boost

In [24]:
catboost_model = CatBoostClassifier(
    iterations=500,
    learning_rate=0.1,
    depth=6,
    loss_function='Logloss',
    l2_leaf_reg=2.599810346285361e-20,
    random_seed=RANDOM_STATE,
    verbose=0
)

catboost_model.fit(X_train_final, y_train_final)

# # 검증 데이터에서 예측 수행
# y_pred_catboost = catboost_model.predict(X_val)


In [25]:
# # F1 Score 계산
# f1 = f1_score(y_val, y_pred_catboost, average='binary', pos_label=0)
# print(f'Validation F1 Score: {f1}')

# soft voting

In [26]:
# 소프트 보팅을 위한 VotingClassifier 정의
soft_voting_model = VotingClassifier(
    estimators=[
        ('bagging', cart_bagging_model_reduced), 
        ('lgbm', best_lgb_model),
        ('catboost', catboost_model)  # CatBoost 모델 추가
    ],
    voting='soft'
)

# VotingClassifier로 학습 수행
soft_voting_model.fit(X_train_final, y_train_final)

# # 검증 데이터에서 예측 수행
# y_pred_soft_voting = soft_voting_model.predict(X_val)

[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 38156, number of negative: 37756
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.044546 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 15497
[LightGBM] [Info] Number of data points in the train set: 75912, number of used features: 135
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with posit

[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f

[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f

[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f

[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f

[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f

[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f

VotingClassifier(estimators=[('bagging',
                              BaggingClassifier(estimator=DecisionTreeClassifier(random_state=42),
                                                n_estimators=55,
                                                random_state=42)),
                             ('lgbm',
                              LGBMClassifier(class_weight='balanced',
                                             colsample_bytree=0.7612130831440033,
                                             learning_rate=0.021711635900896083,
                                             max_depth=10, min_child_samples=5,
                                             n_estimators=800, num_leaves=366,
                                             random_state=110,
                                             reg_alpha=0.6540453950629128,
                                             reg_lambda=3.226039925574031,
                                             subsample=0.7833554249870598)),
                             ('catboost',
                              <catboost.core.CatBoostClassifier object at 0x7f9385d2fe50>)],
                 voting='soft')

In [27]:
# # F1 Score 계산
# f1_soft_voting = f1_score(y_val, y_pred_soft_voting, average='binary', pos_label=0)
# print(f'Soft Voting Validation F1 Score: {f1_soft_voting}')

# 테스트 데이터

In [28]:
# 모델로부터 테스트 데이터에 대한 예측 확률을 얻음
test_pred_proba_voting = soft_voting_model.predict_proba(X_test_encoded)[:, 1]  # 양성 클래스(1)에 대한 확률
optimal_threshold = 0.78
test_predictions_threshold = np.where(test_pred_proba_voting >= optimal_threshold, 1, 0)

# test_predictions_threshold가 임계값을 적용한 최종 예측 결과
unique, counts = np.unique(test_predictions_threshold, return_counts=True)
# 1과 0의 개수를 출력
print(f"Number of 1's: {counts[unique == 1][0]}")
print(f"Number of 0's: {counts[unique == 0][0]}")

Number of 1's: 16505
Number of 0's: 856


In [29]:
# 예측 결과를 AbNormal과 Normal로 변환
target_labels = target_encoder.inverse_transform(test_predictions_threshold)

# 제출 데이터 읽어오기 (df_test는 전처리된 데이터가 저장됨)
df_sub = pd.read_csv("submission.csv")
df_sub["target"] = target_labels


# 제출 파일 저장
df_sub.to_csv("submission.csv", index=False)

print("Submission file created successfully.")

Submission file created successfully.


# 참고 코드

In [30]:
# import numpy as np
# from sklearn.metrics import precision_score, recall_score, f1_score

# #predict_proba의 확률이 임계값 이상이면 1(Normal), 아니면 0(AbNormal)로 반환
# y_scores = soft_voting_model.predict_proba(X_val)[:, 1]

# # 임계값 범위 설정 (0.4부터 1.0까지 0.05 간격으로)
# thresholds = np.arange(0.6, 1.05, 0.025)

# # 각 임계값에 대한 Precision, Recall, F1 Score (positive=AbNormal, 0) 계산 및 출력
# for threshold in thresholds:
#     y_pred_threshold = np.where(y_scores >= threshold, 1, 0)
    
#     precision = precision_score(y_val, y_pred_threshold, pos_label=0)  # 음성(0) Precision
#     recall = recall_score(y_val, y_pred_threshold, pos_label=0)  # 음성(0) Recall
#     f1 = f1_score(y_val, y_pred_threshold, pos_label=0)  # 음성(0) F1 Score
    
#     print(f'Threshold: {threshold:.2f} - Precision: {precision:.4f}, Recall: {recall:.4f}, F1 Score: {f1:.4f}')
   